## 🏢 3. MCP Integration

**Goal**: Connect agents to MCP Tools.

MCP (Model Context Protocol) servers are specialized programs that let AI models securely connect to external data sources and tools through a standardized protocol

In [1]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from utils.config import get_foundry_client, get_credential, get_env
from agent_framework import Agent, tool

client = get_foundry_client()
print("✅ Connected to Foundry")

c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


2026-10-23 07:26:54,845 - azure.ai.agentserver - INFO - Application Insights not configured, telemetry export disabled.
✅ Connected to Foundry


---
### Part A — Remote MCP Server 🌐
Connect to a remote MCP server (e.g. Microsoft Learn) managed by Foundry (approval, allowed tools etc).

In [2]:
# ── A1. Create agent with MCP tool ───────────────────────────────────────────
MCP_URL = get_env("MCP_SERVER_URL", required=False)

if not MCP_URL or MCP_URL.startswith("<"):
    print("⚠️  MCP_SERVER_URL not set — skipping MCP demo.")
else:
    learn_mcp = client.get_mcp_tool(
        name="Microsoft Learn MCP",
        url=MCP_URL,
        approval_mode="never_require",
    )

    mcp_agent = Agent(
        client=client,
        name="MicrosoftLearnAgent",
        instructions=(
            "You are a technical documentation assistant. Use the MCP tool to "
            "search Microsoft Learn and provide accurate, up-to-date guidance "
            "with links to official docs."
        ),
        # Agent expects tool specs in parseable form (dict/FunctionTool/etc.).
        tools=[learn_mcp.as_dict()],
    )
    print(f"🤖 MCP agent: {mcp_agent.name}")

🤖 MCP agent: MicrosoftLearnAgent


In [3]:
# ── A2. Ask a documentation question ─────────────────────────────────────────
if MCP_URL and not MCP_URL.startswith("<"):
    result = await mcp_agent.run(
        "How do I create an Azure Function with a timer trigger in Python?"
    )
    print(f"🗨️  Agent: {result}")

🗨️  Agent: To create an Azure Function with a timer trigger in Python, follow these steps:

### Overview

Azure Functions support timer triggers to execute functions on a schedule. The following instructions apply to both Python v1 and v2 programming models.

### Prerequisites

- [Azure Functions Core Tools](https://learn.microsoft.com/azure/azure-functions/functions-run-local?pivots=programming-language-python#install-the-azure-functions-core-tools)
- [Python 3.x](https://www.python.org/downloads/)
- [Azurite storage emulator](https://learn.microsoft.com/azure/storage/common/storage-use-azurite)
- [Visual Studio Code with Azure Functions and Python extensions](https://learn.microsoft.com/azure/developer/azure-developer-cli/install-azd)

### Steps to Create and Run a Timer Trigger Function

1. **Create a Local Project**:
    - Open your terminal and run:
      ```bash
      func init MyFunctionProject --python
      ```
    - Navigate to the project folder:
      ```bash
      cd MyFun

---
### Part B — Agent-as-Tool Composition 🔗
Use one agent as a tool for another — enabling agent delegation without a full workflow.

In [4]:
# ── C1. Create specialist agents ─────────────────────────────────────────────
from typing import Annotated
from pydantic import Field

# A "math expert" agent
math_agent = client.as_agent(
    name="MathExpert",
    description="An agent that solves math problems step by step.",
    instructions="You are a math expert. Solve problems step by step with clear explanations.",
    # Pass Azure SDK tool models as dict specs so Agent tool parsing is clean.
    tools=[client.get_code_interpreter_tool().as_dict()],
)

# A custom function tool for CRM lookup
@tool(description="Look up customer info by email.")
def lookup_customer(
    email: Annotated[str, Field(description="Customer email address")],
) -> str:
    customers = {
        "alice@contoso.com": {"name": "Alice Smith", "plan": "Enterprise", "tickets": 3},
        "bob@contoso.com": {"name": "Bob Jones", "plan": "Starter", "tickets": 12},
    }
    info = customers.get(email.lower())
    if not info:
        return f"No customer found with email: {email}"
    return str(info)

print("🔬 Specialist agents and tools created")

🔬 Specialist agents and tools created


In [5]:
# ── C2. Create coordinator that uses specialists as tools ────────────────────
coordinator = Agent(
    client=client,
    name="Coordinator",
    instructions=(
        "You are a helpful coordinator. You have access to:\n"
        "- A math expert for calculations and data analysis\n"
        "- A customer lookup tool for CRM queries\n"
        "Route requests to the appropriate tool and present unified answers."
    ),
    tools=[
        math_agent.as_tool(),       # agent used as a tool!
        lookup_customer,            # function tool
    ],
)
print(f"🎯 Coordinator agent: {coordinator.name}")
print("   Tools: MathExpert (agent-as-tool), lookup_customer (function tool)")

🎯 Coordinator agent: Coordinator
   Tools: MathExpert (agent-as-tool), lookup_customer (function tool)


In [6]:
# ── C3. Test routing — math question delegates to MathExpert ─────────────────
result = await coordinator.run(
    "What is the compound annual growth rate if revenue grew from $1M to $2.5M over 3 years?"
)
print(f"🗨️  Coordinator: {result}")

🗨️  Coordinator: The Compound Annual Growth Rate (CAGR) for the revenue growth from $1M to $2.5M over 3 years is approximately 35.72%.


In [7]:
# ── C4. Test routing — CRM query uses function tool ──────────────────────────
result = await coordinator.run(
    "1 Ticket take 2 hours to fix. Look up the customer info for bob@contoso.com and calculate how much time it will take fix the issues. If everyone works 8 hours a day then how many days will it take"
)
print(f"🗨️  Coordinator: {result}")

🗨️  Coordinator: Here is the information you requested:

- **Customer Info:** Bob Jones, who has a Starter plan and currently has 12 open tickets.
- **Time Calculation:** 
  - The time required to fix 12 tickets (assuming each ticket takes 2 hours) is:
    \[
    \text{Total hours} = 12 \times 2 = 24 \, \text{hours}
    \]
  - Working 8 hours a day means:
    \[
    \text{Number of days} = \frac{24}{8} = 3 \, \text{days}
    \]

Therefore, it will take 3 full days to resolve all of Bob's issues.


---
## ✅ What you demonstrated
| Capability | Agent Framework Feature |
|---|---|
| MCP integration | `client.get_mcp_tool(name=..., url=...)` |
| Agent-as-tool | `agent.as_tool()` — one agent calls another |
| Mixed tool types | Function tools + agent tools on same coordinator |

**Next →** [05_multi_agent_orchestration_graph.ipynb](./05_multi_agent_orchestration_graph.ipynb) — graph-based Workflows with `WorkflowBuilder`.